# 07 — Two more seeds of mixed inverted-Mel

The checkpoint already in the paper table is seed 42, at lfcc_vs_mel_compare/runs/inverted_mel/. This notebook does not train that folder again.

It trains inverted-Mel twice more, seeds 43 and 44, with the same recipe: 15 epochs, patience 4, batch size 8, full corpus-balanced 2017+PA train set. Each seed uses its own output folder. The seed also redraws the speaker-disjoint validation split.

After each training run it scores that checkpoint on 2017 eval and PA eval. Expect about two to two and a half hours.


In [ ]:
import json
import sys
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "common.py").exists():
    HERE = Path(r"D:\\speaker-verification-system\\replay-cnn-baseline\\experiments\\paper_official_eval")
sys.path.insert(0, str(HERE))

from common import asv17_root, ensure_pa_ready, experiments_dir, pa_root, paper_runs

COMPARE = experiments_dir() / "lfcc_vs_mel_compare"
sys.path.insert(0, str(COMPARE))
from experiment_lib import eval_checkpoint_on_corpora, train_feature

ensure_pa_ready()

SEED42 = COMPARE / "runs" / "inverted_mel" / "best_inverted_mel_mixed_2017_pa2019.pt"
if not SEED42.is_file():
    raise FileNotFoundError(SEED42)

rows = []
for seed in (43, 44):
    out = COMPARE / "runs" / f"inverted_mel_seed{seed}"
    print(f"\n===== train inverted_mel seed {seed} -> {out} =====", flush=True)
    summary = train_feature(
        "inverted_mel",
        asv17_root=asv17_root(),
        pa_root=pa_root(),
        output_dir=out,
        epochs=15,
        patience=4,
        batch_size=8,
        max_train=0,
        max_val=0,
        seed=seed,
        refresh_cache=False,
    )
    ckpt = Path(summary["checkpoint"]).resolve()
    if ckpt == SEED42.resolve():
        raise RuntimeError("refusing to overwrite the seed-42 checkpoint")
    eval_out = paper_runs() / f"mixed_imel_seed{seed}_eval"
    print(f"===== score seed {seed} on official eval lists =====", flush=True)
    metrics = eval_checkpoint_on_corpora(
        ckpt,
        asv17_root=asv17_root(),
        pa_root=pa_root(),
        asv17_split="eval",
        pa_split="eval",
        batch_size=16,
        output_dir=eval_out,
        refresh_cache=False,
    )
    row = {
        "seed": seed,
        "checkpoint": str(ckpt),
        "val_eer_percent": summary["best_val_eer_percent"],
        "eer_2017": metrics["asvspoof2017"]["eer_percent"],
        "n_2017": metrics["asvspoof2017"]["n"],
        "eer_pa": metrics["pa2019"]["eer_percent"],
        "n_pa": metrics["pa2019"]["n"],
    }
    rows.append(row)
    print(json.dumps(row, indent=2), flush=True)

table_path = paper_runs() / "mixed_imel_seeds_eval.json"
table_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
print("\nSeed 42 stays at", SEED42)
print("New seeds written to", table_path)
for row in rows:
    print(
        f"seed {row['seed']}: 2017 eval {row['eer_2017']:.2f}% "
        f"({row['n_2017']})  PA eval {row['eer_pa']:.2f}% ({row['n_pa']})"
    )
